In [2]:
#checking the content length of the retrieved scrapper

import json
from pathlib import Path
import re
import sys

#Diagnosting paths first
# notebook_dir = Path('.').resolve()
# print(f"Current working directory: {notebook_dir}")

#Add project root path so we can import src modules
sys.path.append(str(Path('..').resolve()))

#Load all scraped docs
docs = [
    json.loads(f.read_text())
    for f in Path('../data/raw').glob('*.json')
    ]

print(f"Loaded {len(docs)} documents")
for doc in docs:
    print(f" {doc['source']} | {doc['section']} | {doc['content_length']} chars | {doc['title'][:50]}")

Loaded 7 documents
 mparticle | general | 21392 chars | Introduction
 aep | profile | 12028 chars | Real-Time Customer Profile Overview | Adobe Experi
 aep | segmentation | 10154 chars | Segmentation Service Overview | Adobe Experience P
 mparticle | general | 13621 chars | Create an Input
 aep | governance | 6812 chars | Data Governance Overview | Adobe Experience Platfo
 aep | schema | 13432 chars | XDM System Overview | Adobe Experience Platform
 aep | identity | 10595 chars | Identity Service Overview | Adobe Experience Platf


In [5]:
# Analyzing raw content structure. 
 # Test on each source type - AEP, mParticle behave differently
for i, doc in enumerate(docs):
    content = doc['content']

    heading_markers = content.count('\n## ')
    double_newlines = content.count('\n\n')
    single_newlines = content.count('\n')

    print(f"Doc {i}: {doc['source']:10} | {doc['section']:12} | "
          f"## markers: {heading_markers:3} | " 
    f"\\n\\n: {double_newlines:4} | "      
    f"\\n: {single_newlines:4} | "  
    f"chars: {len(content):6}")

print("\n=== FIRST 500 CHARS OF DOC 0 (raw) ===")
print(repr(docs[0]['content'][:500]))

print("\n=== FIRST 500 CHARS OF DOC 1 (raw) ===")
print(repr(docs[1]['content'][:500]))

Doc 0: mparticle  | general      | ## markers:   8 | \n\n:   16 | \n:  331 | chars:  21392
Doc 1: aep        | profile      | ## markers:  17 | \n\n:   33 | \n:   86 | chars:  12028
Doc 2: aep        | segmentation | ## markers:  17 | \n\n:   33 | \n:   96 | chars:  10154
Doc 3: mparticle  | general      | ## markers:   8 | \n\n:   16 | \n:  294 | chars:  13621
Doc 4: aep        | governance   | ## markers:   7 | \n\n:   13 | \n:   58 | chars:   6812
Doc 5: aep        | schema       | ## markers:  12 | \n\n:   24 | \n:   93 | chars:  13432
Doc 6: aep        | identity     | ## markers:   6 | \n\n:   11 | \n:   91 | chars:  10595

=== FIRST 500 CHARS OF DOC 0 (raw) ===
'Data Subject Request API Version 1 and 2\nData Subject Request API Version 3\nPlatform API Overview\nCalculated Attributes\nField Transformations\nWarehouse Sync API Overview\nWarehouse Sync API Tutorial\nWarehouse Sync API Reference\nWarehouse Sync SQL Reference\nWarehouse Sync Troubleshooting Guide\nWarehouse Sync API 

In [16]:
# Deep dive on content strategy
# check where first real heading appears in each doc

print("=== FIRST HEADING POSITION===\n")

for i, doc in enumerate(docs):
    content = doc['content']
    first_heading_pos = content.find('\n## ') # returns the line numbers

    #content before the first heading (navigation noise)
    preamble = content[:first_heading_pos].strip() if first_heading_pos > 0 else ""
    preamble_lines = len(preamble.split('\n')) if preamble else 0 

    print(f"oc {i}: {doc['source']:10} | {doc['section']:12}) | " 
          f"First ## at char: {first_heading_pos:5} | "
          f"preamble_lines: {preamble_lines:3}")
    
    if preamble_lines > 5:
        print( "Navigation noise detected: first 100 chars of preamble:")
        print(f"  {repr(preamble[:100])}")
    print()


#Show what a clean AEP section looks like vs mParticle

print("===AEP section sample(Doc 1)===")
aep_doc = docs[1]
sections_aep = aep_doc['content'].split('\n## ')
print(f"Section count: {len(sections_aep)}")

for s in sections_aep[1:4]: # skip preamble show 3 sections
    lines = s.strip().split('\n')
    heading = lines[0] if lines else ""
    body_preview = ' '.join(lines[1:])[:200] if len(lines) > 1 else ""
    print(f"\n ## {heading}")
    print(f" {body_preview}")


#mParticle section
print("===mParticle section sample(Doc 0)===")
mp_doc = docs[0]
sections_mp = mp_doc['content'].split('\n## ')
print(f"Section count: {len(sections_mp)}")

for s in sections_mp[1:4]: # skip preamble show 3 sections
    lines = s.strip().split('\n')
    heading = lines[0] if lines else ""
    body_preview = ' '.join(lines[1:])[:200] if len(lines) > 1 else ""
    print(f"\n ## {heading}")
    print(f" {body_preview}")


=== FIRST HEADING POSITION===

oc 0: mparticle  | general     ) | First ## at char:  7750 | preamble_lines: 224
Navigation noise detected: first 100 chars of preamble:
  'Data Subject Request API Version 1 and 2\nData Subject Request API Version 3\nPlatform API Overview\nCa'

oc 1: aep        | profile     ) | First ## at char:     0 | preamble_lines:   0

oc 2: aep        | segmentation) | First ## at char:     0 | preamble_lines:   0

oc 3: mparticle  | general     ) | First ## at char:  8467 | preamble_lines: 231
Navigation noise detected: first 100 chars of preamble:
  'Data Subject Request API Version 1 and 2\nData Subject Request API Version 3\nPlatform API Overview\nCa'

oc 4: aep        | governance  ) | First ## at char:     0 | preamble_lines:   0

oc 5: aep        | schema      ) | First ## at char:  1217 | preamble_lines:   4

oc 6: aep        | identity    ) | First ## at char:     0 | preamble_lines:   0

===AEP section sample(Doc 1)===
Section count: 18

 ## Real-Time Cu

In [ ]:
# Experiment heading based chunking. 